In [3]:
!unzip train.zip -d train/
!unzip test.zip -d test/

unzip:  cannot find or open train.zip, train.zip.zip or train.zip.ZIP.
unzip:  cannot find or open test.zip, test.zip.zip or test.zip.ZIP.


In [4]:
from __future__ import absolute_import, division, print_function, unicode_literals
%tensorflow_version 2.x
import tensorflow as tf
%load_ext tensorboard
import datetime
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import os

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


In [5]:
def load_and_preprocess_image(file_path):
    filename = tf.strings.split(file_path, os.sep)[-1]
    parts = tf.strings.split(filename, '.')
    class_name = parts[0]
    label = tf.cast(tf.equal(class_name, 'dog'), tf.int32)

    image = tf.io.read_file(file_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [128, 128])
    image = tf.cast(image, tf.float32) / 255.0

    return image, label

In [6]:
import os

BATCH_SIZE = 10
train_files = tf.data.Dataset.list_files("/content/drive/My Drive/train/*.jpg", shuffle=True)
test_files = tf.data.Dataset.list_files("/content/drive/My Drive/test/*.jpg", shuffle=False)


train_dataset = train_files.map(load_and_preprocess_image).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_files.map(load_and_preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

InvalidArgumentError: Expected 'tf.Tensor(False, shape=(), dtype=bool)' to be true. Summarized data: b'No files matched pattern: /content/drive/My Drive/train/*.jpg'

In [ ]:
logdir = "logs/scalars/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
%tensorboard --logdir logs
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir, update_freq='batch')

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False

In [ ]:
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.Dense(2, activation='relu')
])

In [ ]:
base_model.trainable = True
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

history = model.fit(train_dataset, epochs=5, validation_data=test_dataset, callbacks=[tensorboard_callback])

In [ ]:
model.summary()

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset)
print("Test Accuracy:", test_acc)

In [ ]:
img_path = "/content/test/0.1047.jpg"


img = tf.io.read_file(img_path)
img = tf.image.decode_jpeg(img, channels=3)
img = tf.image.resize(img, [128, 128])
img = tf.cast(img, tf.float32) / 255.0

plt.imshow(img.numpy())
plt.axis('off')
plt.title("Test Image")
plt.show()

pred = model.predict(tf.expand_dims(img, axis=0))
class_names = ['cat', 'dog']
print("Predicted label:", class_names[tf.argmax(pred[0])])